In [16]:
import torch
import torch.nn as nn
import torchvision.models as tv_models

from torchvision.models import (
    ResNet18_Weights,
    DenseNet121_Weights,
    EfficientNet_B0_Weights,
)

In [17]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.classifier = nn.Linear(
            256,
            num_classes
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

In [18]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.classifier = nn.Linear(
            256,
            num_classes
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

In [19]:
def build_model(
    model_name,
    num_classes=2,
    pretrained=True
):
    model_name = model_name.lower()

    if model_name == "small_cnn":
        model = SmallCNN(
            num_classes=num_classes
        )

    elif model_name == "resnet18":
        weights = (
            ResNet18_Weights.DEFAULT
            if pretrained
            else None
        )

        model = tv_models.resnet18(
            weights=weights
        )

        model.fc = nn.Linear(
            model.fc.in_features,
            num_classes
        )

    elif model_name == "densenet121":
        weights = (
            DenseNet121_Weights.DEFAULT
            if pretrained
            else None
        )

        model = tv_models.densenet121(
            weights=weights
        )

        model.classifier = nn.Linear(
            model.classifier.in_features,
            num_classes
        )

    elif model_name == "efficientnet_b0":
        weights = (
            EfficientNet_B0_Weights.DEFAULT
            if pretrained
            else None
        )

        model = tv_models.efficientnet_b0(
            weights=weights
        )

        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            num_classes
        )

    else:
        raise ValueError(
            f"未知模型：{model_name}"
        )

    return model

In [20]:
model = build_model(
    "small_cnn",
    num_classes=2,
    pretrained=False
)

In [21]:
model = build_model(
    "resnet18",
    num_classes=2,
    pretrained=True
)

In [22]:
model = build_model(
    "densenet121",
    num_classes=2,
    pretrained=True
)

In [23]:
model = build_model(
    "efficientnet_b0",
    num_classes=2,
    pretrained=True
)

In [24]:
device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)

In [6]:
import numpy as np
from torchvision.datasets import ImageFolder

train_dataset = ImageFolder(
    f"{DATA_DIR}/train"
)

targets = np.asarray(
    train_dataset.targets
)

counts = np.bincount(targets)
majority_class = int(counts.argmax())

print("类别：", train_dataset.class_to_idx)
print("训练集类别数量：", counts)
print("多数类别：", majority_class)

类别： {'negative': 0, 'positive': 1}
训练集类别数量： [1050 1050]
多数类别： 0


In [7]:
import numpy as np

from pathlib import Path
from torchvision.datasets import ImageFolder


# ==========================================
# 选择任务：
# "tumor"：脑肿瘤阴性/阳性
# "hemorrhage"：脑出血阴性/阳性
# ==========================================
TASK = "tumor"


if TASK == "tumor":
    DATA_DIR = (
        "/root/autodl-tmp/medical_data/"
        "brain_tumor"
    )

elif TASK == "hemorrhage":
    DATA_DIR = (
        "/root/autodl-tmp/medical_data/"
        "brain_hemorrhage"
    )

else:
    raise ValueError(
        "TASK只能是tumor或hemorrhage"
    )


data_path = Path(DATA_DIR)

if not data_path.is_dir():
    raise FileNotFoundError(
        f"找不到数据目录：{DATA_DIR}"
    )


train_dataset = ImageFolder(
    data_path / "train"
)

val_dataset = ImageFolder(
    data_path / "val"
)

test_dataset = ImageFolder(
    data_path / "test"
)


print("=" * 55)
print(" Majority基线")
print("=" * 55)
print("任务：", TASK)
print("数据目录：", DATA_DIR)
print("类别映射：", train_dataset.class_to_idx)
print("训练集：", len(train_dataset))
print("验证集：", len(val_dataset))
print("测试集：", len(test_dataset))
print("=" * 55)

 Majority基线
任务： tumor
数据目录： /root/autodl-tmp/medical_data/brain_tumor
类别映射： {'negative': 0, 'positive': 1}
训练集： 2100
验证集： 450
测试集： 450


In [8]:
train_targets = np.asarray(
    train_dataset.targets
)

class_counts = np.bincount(
    train_targets,
    minlength=len(train_dataset.classes)
)

majority_class_index = int(
    class_counts.argmax()
)

majority_class_name = (
    train_dataset.classes[
        majority_class_index
    ]
)

print("训练集类别数量：")

for index, class_name in enumerate(
    train_dataset.classes
):
    print(
        f"  {class_name}: "
        f"{class_counts[index]}"
    )

print()
print(
    "多数类别：",
    majority_class_name
)

训练集类别数量：
  negative: 1050
  positive: 1050

多数类别： negative


In [9]:
def majority_accuracy(
    dataset,
    majority_class_index
):
    true_labels = np.asarray(
        dataset.targets
    )

    predictions = np.full(
        shape=len(true_labels),
        fill_value=majority_class_index,
        dtype=np.int64
    )

    accuracy = (
        predictions == true_labels
    ).mean()

    return accuracy


val_accuracy = majority_accuracy(
    val_dataset,
    majority_class_index
)

test_accuracy = majority_accuracy(
    test_dataset,
    majority_class_index
)


print("=" * 55)
print(" Majority实验结果")
print("=" * 55)
print(
    f"预测策略：所有图片均预测为 "
    f"{majority_class_name}"
)
print(
    f"Validation Accuracy："
    f"{val_accuracy * 100:.2f}%"
)
print(
    f"Test Accuracy      ："
    f"{test_accuracy * 100:.2f}%"
)
print("=" * 55)

 Majority实验结果
预测策略：所有图片均预测为 negative
Validation Accuracy：50.00%
Test Accuracy      ：50.00%


In [23]:
import os
import sys
import random
import time
import copy

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import torchvision.models as tv_models

from torchvision.models import (
    ResNet18_Weights,
    DenseNet121_Weights,
    EfficientNet_B0_Weights,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

from IPython.display import display


# ==========================================
# 选择任务
# "tumor"：脑肿瘤阴/阳性
# "hemorrhage"：脑出血阴/阳性
# ==========================================
TASK = "tumor"


if TASK == "tumor":
    DATA_DIR = (
        "/root/autodl-tmp/medical_data/"
        "brain_tumor"
    )
    BATCH_SIZE = 64

elif TASK == "hemorrhage":
    DATA_DIR = (
        "/root/autodl-tmp/medical_data/"
        "brain_hemorrhage"
    )
    BATCH_SIZE = 16

else:
    raise ValueError(
        "TASK只能是tumor或hemorrhage"
    )


RESULT_ROOT = Path(
    "/root/autodl-tmp/baseline_results"
) / TASK

RESULT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


MODEL_NAMES = [
    "small_cnn",
    "resnet18",
    "densenet121",
    "efficientnet_b0",
]

SEEDS = [1, 3, 7, 21, 42]

NUM_CLASSES = 2
EPOCHS = 30
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
NUM_WORKERS = 4
GPU = 0

device = torch.device(
    f"cuda:{GPU}"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 60)
print("多模型、多随机种子分类实验")
print("=" * 60)
print("任务：", TASK)
print("数据：", DATA_DIR)
print("模型：", MODEL_NAMES)
print("随机种子：", SEEDS)
print("Epochs：", EPOCHS)
print("Batch size：", BATCH_SIZE)
print("设备：", device)
print("输出目录：", RESULT_ROOT)
print("=" * 60)

多模型、多随机种子分类实验
任务： tumor
数据： /root/autodl-tmp/medical_data/brain_tumor
模型： ['small_cnn', 'resnet18', 'densenet121', 'efficientnet_b0']
随机种子： [1, 3, 7, 21, 42]
Epochs： 30
Batch size： 64
设备： cuda:0
输出目录： /root/autodl-tmp/baseline_results/tumor


In [24]:

train_transform = transforms.Compose([
    transforms.Grayscale(
        num_output_channels=3
    ),
    transforms.Resize((224, 224)),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

evaluation_transform = transforms.Compose([
    transforms.Grayscale(
        num_output_channels=3
    ),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [25]:
class SmallCNN(nn.Module):
    def __init__(
        self,
        num_classes=2
    ):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                3, 32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(
                32, 64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(
                64, 128,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(
                128, 256,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.classifier = nn.Linear(
            256,
            num_classes
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def build_model(
    model_name,
    num_classes=2
):
    if model_name == "small_cnn":
        model = SmallCNN(
            num_classes=num_classes
        )

    elif model_name == "resnet18":
        model = tv_models.resnet18(
            weights=ResNet18_Weights.DEFAULT
        )

        model.fc = nn.Linear(
            model.fc.in_features,
            num_classes
        )

    elif model_name == "densenet121":
        model = tv_models.densenet121(
            weights=DenseNet121_Weights.DEFAULT
        )

        model.classifier = nn.Linear(
            model.classifier.in_features,
            num_classes
        )

    elif model_name == "efficientnet_b0":
        model = tv_models.efficientnet_b0(
            weights=EfficientNet_B0_Weights.DEFAULT
        )

        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            num_classes
        )

    else:
        raise ValueError(
            f"未知模型：{model_name}"
        )

    return model

In [26]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = (
        torch.initial_seed() % 2**32
    )

    np.random.seed(worker_seed)
    random.seed(worker_seed)


def create_loaders(seed):
    train_dataset = ImageFolder(
        Path(DATA_DIR) / "train",
        transform=train_transform
    )

    val_dataset = ImageFolder(
        Path(DATA_DIR) / "val",
        transform=evaluation_transform
    )

    test_dataset = ImageFolder(
        Path(DATA_DIR) / "test",
        transform=evaluation_transform
    )

    if (
        train_dataset.class_to_idx
        != val_dataset.class_to_idx
        or train_dataset.class_to_idx
        != test_dataset.class_to_idx
    ):
        raise ValueError(
            "train/val/test类别映射不一致"
        )

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=generator,
        drop_last=False,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    return (
        train_loader,
        val_loader,
        test_loader,
        train_dataset.class_to_idx,
    )

In [27]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer
):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(images)
        loss = criterion(
            logits,
            labels
        )

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        total_loss += (
            loss.item() * batch_size
        )

        total_correct += (
            logits.argmax(dim=1) == labels
        ).sum().item()

        total_samples += batch_size

    return {
        "loss": total_loss / total_samples,
        "accuracy": (
            total_correct / total_samples
        ),
    }

In [28]:
@torch.no_grad()
def evaluate_model(
    model,
    loader,
    criterion
):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    true_labels = []
    predictions = []
    positive_probabilities = []

    for images, labels in loader:
        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        logits = model(images)
        loss = criterion(
            logits,
            labels
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        predicted = logits.argmax(dim=1)

        batch_size = labels.size(0)

        total_loss += (
            loss.item() * batch_size
        )
        total_samples += batch_size

        true_labels.extend(
            labels.cpu().numpy().tolist()
        )

        predictions.extend(
            predicted.cpu().numpy().tolist()
        )

        positive_probabilities.extend(
            probabilities[:, 1]
            .cpu()
            .numpy()
            .tolist()
        )

    true_labels = np.asarray(true_labels)
    predictions = np.asarray(predictions)
    positive_probabilities = np.asarray(
        positive_probabilities
    )

    matrix = confusion_matrix(
        true_labels,
        predictions,
        labels=[0, 1]
    )

    tn, fp, fn, tp = matrix.ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    try:
        auc = roc_auc_score(
            true_labels,
            positive_probabilities
        )
    except ValueError:
        auc = np.nan

    return {
        "loss": total_loss / total_samples,
        "accuracy": accuracy_score(
            true_labels,
            predictions
        ),
        "balanced_accuracy": (
            balanced_accuracy_score(
                true_labels,
                predictions
            )
        ),
        "precision": precision_score(
            true_labels,
            predictions,
            zero_division=0
        ),
        "sensitivity": recall_score(
            true_labels,
            predictions,
            zero_division=0
        ),
        "specificity": specificity,
        "f1": f1_score(
            true_labels,
            predictions,
            zero_division=0
        ),
        "roc_auc": auc,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

In [29]:
all_results = []
all_histories = {}

for model_name in MODEL_NAMES:
    for run_index, seed in enumerate(
        SEEDS,
        start=1
    ):
        print()
        print("=" * 72)
        print(
            f"模型：{model_name} | "
            f"Seed：{seed} | "
            f"Run：{run_index}/{len(SEEDS)}"
        )
        print("=" * 72)

        set_seed(seed)

        (
            train_loader,
            val_loader,
            test_loader,
            class_to_idx,
        ) = create_loaders(seed)

        model = build_model(
            model_name,
            num_classes=NUM_CLASSES
        ).to(device)

        criterion = nn.CrossEntropyLoss()

        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=LEARNING_RATE,
            momentum=0.9,
            weight_decay=WEIGHT_DECAY
        )

        scheduler = (
            torch.optim.lr_scheduler
            .CosineAnnealingLR(
                optimizer,
                T_max=EPOCHS
            )
        )

        run_dir = (
            RESULT_ROOT /
            model_name /
            f"seed_{seed}"
        )

        run_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        checkpoint_path = (
            run_dir /
            "model_best.pth.tar"
        )

        history = []
        best_val_accuracy = -1.0
        best_epoch = -1

        start_time = time.time()

        for epoch in range(
            1,
            EPOCHS + 1
        ):
            train_metrics = train_one_epoch(
                model,
                train_loader,
                criterion,
                optimizer
            )

            val_metrics = evaluate_model(
                model,
                val_loader,
                criterion
            )

            current_lr = (
                optimizer.param_groups[0]["lr"]
            )

            history.append({
                "Epoch": epoch,
                "Learning Rate": current_lr,
                "Train Loss": (
                    train_metrics["loss"]
                ),
                "Train Accuracy": (
                    train_metrics["accuracy"]
                ),
                "Validation Loss": (
                    val_metrics["loss"]
                ),
                "Validation Accuracy": (
                    val_metrics["accuracy"]
                ),
                "Validation F1": (
                    val_metrics["f1"]
                ),
                "Validation AUC": (
                    val_metrics["roc_auc"]
                ),
            })

            improved = (
                val_metrics["accuracy"]
                > best_val_accuracy
            )

            if improved:
                best_val_accuracy = (
                    val_metrics["accuracy"]
                )
                best_epoch = epoch

                torch.save(
                    {
                        "model_name": model_name,
                        "seed": seed,
                        "epoch": epoch,
                        "state_dict": (
                            model.state_dict()
                        ),
                        "optimizer": (
                            optimizer.state_dict()
                        ),
                        "val_accuracy": (
                            best_val_accuracy
                        ),
                        "class_to_idx": (
                            class_to_idx
                        ),
                        "data_dir": DATA_DIR,
                        "task": TASK,
                        "num_classes": (
                            NUM_CLASSES
                        ),
                    },
                    checkpoint_path
                )

            scheduler.step()

            print(
                f"Epoch {epoch:02d}/{EPOCHS} | "
                f"Train "
                f"{train_metrics['accuracy'] * 100:6.2f}% | "
                f"Val "
                f"{val_metrics['accuracy'] * 100:6.2f}% | "
                f"Val F1 "
                f"{val_metrics['f1']:.4f} | "
                f"{'* Best' if improved else ''}"
            )

        # 重新加载验证集最优模型
        checkpoint = torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=False
        )

        model.load_state_dict(
            checkpoint["state_dict"]
        )

        test_metrics = evaluate_model(
            model,
            test_loader,
            criterion
        )

        elapsed_minutes = (
            time.time() - start_time
        ) / 60

        history_df = pd.DataFrame(history)

        history_path = (
            run_dir /
            "training_history.csv"
        )

        history_df.to_csv(
            history_path,
            index=False
        )

        all_histories[
            (model_name, seed)
        ] = history_df

        result = {
            "Task": TASK,
            "Model": model_name,
            "Seed": seed,
            "Best Epoch": best_epoch,
            "Best Validation Accuracy": (
                best_val_accuracy * 100
            ),
            "Test Accuracy": (
                test_metrics["accuracy"] * 100
            ),
            "Balanced Accuracy": (
                test_metrics[
                    "balanced_accuracy"
                ] * 100
            ),
            "Sensitivity": (
                test_metrics[
                    "sensitivity"
                ] * 100
            ),
            "Specificity": (
                test_metrics[
                    "specificity"
                ] * 100
            ),
            "Precision": (
                test_metrics[
                    "precision"
                ] * 100
            ),
            "F1": (
                test_metrics["f1"] * 100
            ),
            "ROC-AUC": (
                test_metrics[
                    "roc_auc"
                ] * 100
            ),
            "TN": test_metrics["tn"],
            "FP": test_metrics["fp"],
            "FN": test_metrics["fn"],
            "TP": test_metrics["tp"],
            "Minutes": elapsed_minutes,
            "Checkpoint": str(
                checkpoint_path
            ),
        }

        all_results.append(result)

        print()
        print(
            f"完成：{model_name}, seed={seed}"
        )
        print(
            f"最佳Epoch：{best_epoch}"
        )
        print(
            f"最佳验证准确率："
            f"{best_val_accuracy * 100:.2f}%"
        )
        print(
            f"最终测试准确率："
            f"{test_metrics['accuracy'] * 100:.2f}%"
        )
        print(
            f"ROC-AUC："
            f"{test_metrics['roc_auc'] * 100:.2f}%"
        )
        print(
            f"模型已保存：{checkpoint_path}"
        )

        del model
        torch.cuda.empty_cache()


模型：small_cnn | Seed：1 | Run：1/5
Epoch 01/30 | Train  64.10% | Val  63.56% | Val F1 0.5030 | * Best
Epoch 02/30 | Train  70.24% | Val  75.11% | Val F1 0.7489 | * Best
Epoch 03/30 | Train  73.19% | Val  75.78% | Val F1 0.7666 | * Best
Epoch 04/30 | Train  75.14% | Val  76.22% | Val F1 0.7585 | * Best
Epoch 05/30 | Train  77.43% | Val  80.67% | Val F1 0.8009 | * Best
Epoch 06/30 | Train  78.19% | Val  80.44% | Val F1 0.8167 | 
Epoch 07/30 | Train  79.29% | Val  80.44% | Val F1 0.8189 | 
Epoch 08/30 | Train  79.57% | Val  71.33% | Val F1 0.7725 | 
Epoch 09/30 | Train  78.71% | Val  84.22% | Val F1 0.8360 | * Best
Epoch 10/30 | Train  80.52% | Val  81.33% | Val F1 0.8250 | 
Epoch 11/30 | Train  80.81% | Val  80.22% | Val F1 0.7866 | 
Epoch 12/30 | Train  80.90% | Val  82.67% | Val F1 0.8319 | 
Epoch 13/30 | Train  81.76% | Val  82.44% | Val F1 0.8454 | 
Epoch 14/30 | Train  80.62% | Val  82.44% | Val F1 0.8294 | 
Epoch 15/30 | Train  81.71% | Val  79.33% | Val F1 0.8194 | 
Epoch 16/30 | Tr

In [30]:
results_df = pd.DataFrame(
    all_results
)

detail_path = (
    RESULT_ROOT /
    "all_seed_results.csv"
)

results_df.to_csv(
    detail_path,
    index=False
)

display(
    results_df.style.format({
        "Best Validation Accuracy": "{:.2f}%",
        "Test Accuracy": "{:.2f}%",
        "Balanced Accuracy": "{:.2f}%",
        "Sensitivity": "{:.2f}%",
        "Specificity": "{:.2f}%",
        "Precision": "{:.2f}%",
        "F1": "{:.2f}%",
        "ROC-AUC": "{:.2f}%",
        "Minutes": "{:.2f}",
    })
)

print("明细已保存：", detail_path)

,Task,Model,Seed,Best Epoch,Best Validation Accuracy,Test Accuracy,Balanced Accuracy,Sensitivity,Specificity,Precision,F1,ROC-AUC,TN,FP,FN,TP,Minutes,Checkpoint
0,tumor,small_cnn,1,24,84.67%,81.78%,81.78%,72.00%,91.56%,89.50%,79.80%,92.17%,206,19,63,162,1.36,/root/autodl-tmp/baseline_results/tumor/small_cnn/seed_1/model_best.pth.tar
1,tumor,small_cnn,3,25,83.56%,83.11%,83.11%,80.44%,85.78%,84.98%,82.65%,92.05%,193,32,44,181,1.35,/root/autodl-tmp/baseline_results/tumor/small_cnn/seed_3/model_best.pth.tar
2,tumor,small_cnn,7,17,83.56%,80.00%,80.00%,84.89%,75.11%,77.33%,80.93%,90.42%,169,56,34,191,1.35,/root/autodl-tmp/baseline_results/tumor/small_cnn/seed_7/model_best.pth.tar
3,tumor,small_cnn,21,13,84.00%,82.89%,82.89%,86.67%,79.11%,80.58%,83.51%,89.98%,178,47,30,195,1.34,/root/autodl-tmp/baseline_results/tumor/small_cnn/seed_21/model_best.pth.tar
4,tumor,small_cnn,42,22,84.22%,82.44%,82.44%,81.78%,83.11%,82.88%,82.33%,91.97%,187,38,41,184,1.34,/root/autodl-tmp/baseline_results/tumor/small_cnn/seed_42/model_best.pth.tar
5,tumor,resnet18,1,16,99.56%,99.56%,99.56%,99.56%,99.56%,99.56%,99.56%,99.87%,224,1,1,224,1.35,/root/autodl-tmp/baseline_results/tumor/resnet18/seed_1/model_best.pth.tar
6,tumor,resnet18,3,25,99.78%,99.33%,99.33%,99.56%,99.11%,99.12%,99.33%,99.89%,223,2,1,224,1.38,/root/autodl-tmp/baseline_results/tumor/resnet18/seed_3/model_best.pth.tar
7,tumor,resnet18,7,9,99.56%,99.56%,99.56%,99.56%,99.56%,99.56%,99.56%,99.92%,224,1,1,224,1.39,/root/autodl-tmp/baseline_results/tumor/resnet18/seed_7/model_best.pth.tar
8,tumor,resnet18,21,20,99.56%,99.78%,99.78%,99.56%,100.00%,100.00%,99.78%,99.96%,225,0,1,224,1.38,/root/autodl-tmp/baseline_results/tumor/resnet18/seed_21/model_best.pth.tar
9,tumor,resnet18,42,6,99.78%,99.56%,99.56%,99.56%,99.56%,99.56%,99.56%,99.70%,224,1,1,224,1.38,/root/autodl-tmp/baseline_results/tumor/resnet18/seed_42/model_best.pth.tar


明细已保存： /root/autodl-tmp/baseline_results/tumor/all_seed_results.csv


In [31]:
metric_columns = [
    "Best Validation Accuracy",
    "Test Accuracy",
    "Balanced Accuracy",
    "Sensitivity",
    "Specificity",
    "Precision",
    "F1",
    "ROC-AUC",
]

summary_rows = []

for model_name in MODEL_NAMES:
    model_results = results_df[
        results_df["Model"] == model_name
    ]

    row = {
        "Model": model_name,
        "Runs": len(model_results),
        "Best Epoch Mean": (
            model_results[
                "Best Epoch"
            ].mean()
        ),
        "Best Epoch Std": (
            model_results[
                "Best Epoch"
            ].std(ddof=1)
        ),
    }

    for metric in metric_columns:
        mean_value = (
            model_results[metric].mean()
        )

        std_value = (
            model_results[metric].std(
                ddof=1
            )
        )

        row[f"{metric} Mean"] = mean_value
        row[f"{metric} Std"] = std_value
        row[f"{metric} Report"] = (
            f"{mean_value:.2f} ± "
            f"{std_value:.2f}"
        )

    summary_rows.append(row)


summary_df = pd.DataFrame(
    summary_rows
)

summary_path = (
    RESULT_ROOT /
    "model_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)

display(
    summary_df[[
        "Model",
        "Runs",
        "Best Epoch Mean",
        "Best Epoch Std",
        "Test Accuracy Report",
        "Sensitivity Report",
        "Specificity Report",
        "F1 Report",
        "ROC-AUC Report",
    ]]
)

print("汇总已保存：", summary_path)

,Model,Runs,Best Epoch Mean,Best Epoch Std,Test Accuracy Report,Sensitivity Report,Specificity Report,F1 Report,ROC-AUC Report
0,small_cnn,5,20.2,5.069517,82.04 ± 1.25,81.16 ± 5.68,82.93 ± 6.29,81.84 ± 1.47,91.32 ± 1.04
1,resnet18,5,15.2,7.791020,99.56 ± 0.16,99.56 ± 0.00,99.56 ± 0.31,99.56 ± 0.16,99.87 ± 0.10
2,densenet121,5,14.6,6.655825,99.24 ± 0.30,99.73 ± 0.24,98.76 ± 0.58,99.25 ± 0.30,99.99 ± 0.01
3,efficientnet_b0,5,19.8,2.774887,99.33 ± 0.16,98.67 ± 0.31,100.00 ± 0.00,99.33 ± 0.16,99.95 ± 0.01


汇总已保存： /root/autodl-tmp/baseline_results/tumor/model_summary.csv


In [ ]:
majority_row = {
    "Model": "majority",
    "Runs": 1,
    "Best Epoch Mean": np.nan,
    "Best Epoch Std": np.nan,
    "Test Accuracy Report": (
        f"{test_accuracy * 100:.2f}%"
    ),
    "Sensitivity Report": "不适用",
    "Specificity Report": "不适用",
    "F1 Report": "不适用",
    "ROC-AUC Report": "50.00%",
}

comparison_table = pd.concat(
    [
        pd.DataFrame([majority_row]),
        summary_df[[
            "Model",
            "Runs",
            "Best Epoch Mean",
            "Best Epoch Std",
            "Test Accuracy Report",
            "Sensitivity Report",
            "Specificity Report",
            "F1 Report",
            "ROC-AUC Report",
        ]]
    ],
    ignore_index=True
)

display(comparison_table)

comparison_path = (
    RESULT_ROOT /
    "final_comparison.csv"
)

comparison_table.to_csv(
    comparison_path,
    index=False
)

print(
    "最终对比表已保存：",
    comparison_path
)